# Orbit Wars RL Model Architecture

Bu notebook Orbit Wars PPO policy/value network mimarisini detaylı açıklar.

Ana kaynak dosyalar:

- `model.py`: Full Set Transformer policy/value network.
- `model_planet_only.py`: Fleet tokenları encoder dışında bırakan planet-only varyant.
- `obs_builder.py`: Model input tensorlarının shape ve semantics'i.
- `rl/ppo.py`: PPO update ve action re-evaluation.
- `rl/rollout.py`: Rollout sırasında action sampling ve transition yazımı.


## 1. Modelin Rolü

Model tek network içinde hem actor hem critic görevi görür. PPO rollout sırasında policy action üretir; update sırasında aynı network eski actionları yeniden evaluate eder.

Forward outputları:

```text
src_logits  : (B, P_max + 1)
tgt_logits  : (B, P_max, P_max)
ship_logits : (B, P_max, n_ship_bins)
value       : (B,)
tokens      : (B, T, d_model)
```

`P_max + 1` içindeki son source slot learned NO-OP slotudur.


## 2. Input Tensorları

Observation builder'dan gelen batch tensorları:

```text
planet_feats : (B, P_max, planet_dim)
fleet_feats  : (B, F_max, fleet_dim)
global_feats : (B, global_dim)
edge_bias    : (B, T, T, edge_dim)
planet_mask  : (B, P_max) bool
fleet_mask   : (B, F_max) bool
source_mask  : (B, P_max) bool
```

Full model defaultları:

```text
P_max = 48
F_max = 64
T = 48 + 64 + 1 = 113
planet_dim = 34
fleet_dim = 11
global_dim = 14
edge_dim = 6
```

Planet-only modelde encoder token count `P_max + 1 = 49` olur.

## 3. Full `OrbitWarsPolicy`

Full model token layout'u:

```text
[planet_0, ..., planet_47, fleet_0, ..., fleet_63, GLOBAL]
```

Ana bileşenler:

| Bileşen | Görev |
|---|---|
| `planet_proj` | Planet feature vektörünü `d_model` tokena projekte eder |
| `fleet_proj` | Fleet feature vektörünü `d_model` tokena projekte eder |
| `global_proj` | Global feature vektörünü global tokena projekte eder |
| `type_embed` | Planet/fleet/global token tipini belirtir |
| `edge_proj` | Pairwise geometry features -> per-head attention bias |
| `TransformerBlock` | Tokenlar arası reasoning |
| `src_head` + `noop_logit` | Source/no-op decision |
| `tgt_query/tgt_key` | Source-conditioned target pointer logits |
| `ship_head` | Ship-bin logits |
| `value_head` | Critic value estimate |


## 4. `PlanetOnlyOrbitWarsPolicy`

Planet-only varyant mevcut traininglerde sık kullanılır. Fleet tokenları encoder'a koymaz; fleet bilgisi planet featurelarındaki incoming summaries üzerinden gelir.

Token layout:

```text
[planet_0, ..., planet_47, GLOBAL]
```

Full modelden farkları:

| Full model | Planet-only model |
|---|---|
| Planet + fleet + global token | Planet + global token |
| `fleet_proj` var | `fleet_proj` yok |
| Value head global token kullanır | Value head global + mean planet pool kullanır |
| Fleet detail attention mümkün | Daha hızlı ve daha az tokenlı |
| Token count 113 | Token count 49 |

Planet-only ayrıca `aux_owner_head` üretir: her planet için mine/enemy/neutral auxiliary logits.


## 5. Projection ve Type Embedding

Raw feature vektörleri önce ortak embedding boyutuna taşınır.

Full model için:

```text
planet_tok = planet_proj(planet_feats)
fleet_tok  = fleet_proj(fleet_feats)
global_tok = global_proj(global_feats).unsqueeze(1)
x = concat([planet_tok, fleet_tok, global_tok], dim=1)
x = x + type_embed(type_ids)
```

Type embedding olmazsa projection sonrası transformer planet/fleet/global token türlerini sadece feature dağılımından çıkarmak zorunda kalır. Type embedding bunu açıkça verir.


## 6. Edge-Biased Attention

Model standart multi-head attention kullanır, ancak attention scorelarına geometric edge bias ekler.

```text
scores = QKᵀ / sqrt(d_head)
scores = scores + projected_edge_bias
attention = softmax(scores)
output = attention V
```

`edge_bias` observation'da `(B, T, T, edge_dim)` gelir. `edge_proj` bunu `(B, T, T, n_heads)` yapar, sonra `(B, n_heads, T, T)` düzenine çevrilir.

Bu bias mesafe, dx/dy, angular fark ve sun crossing bilgisini attention'a doğrudan verir. Orbit Wars'ta launch geometry ve travel time kritik olduğu için bu güçlü bir inductive bias'tır.


In [ ]:
def full_model_shapes(batch=2, p_max=48, f_max=64, d_model=128, heads=4):
    T = p_max + f_max + 1
    d_head = d_model // heads
    return {
        'tokens': (batch, T, d_model),
        'q/k/v': (batch, heads, T, d_head),
        'attention_scores': (batch, heads, T, T),
        'edge_bias_projected': (batch, heads, T, T),
        'src_logits': (batch, p_max + 1),
        'tgt_logits': (batch, p_max, p_max),
        'ship_logits': (batch, p_max, 5),
        'value': (batch,),
    }

full_model_shapes()


## 7. Transformer Block

Her block pre-norm yapıdadır:

```text
x = x + BiasedMHA(LayerNorm(x), attn_bias, key_padding_mask)
x = x + FFN(LayerNorm(x))
```

Feed-forward network:

```text
Linear(d_model, 4*d_model)
GELU
Linear(4*d_model, d_model)
```

Pre-norm transformerlar genelde PPO gibi noisy-gradient traininglerde daha stabildir.


## 8. Padding Mask

Observation padded tokenlar içerir. Transformer bu padded tokenlara attend etmemelidir.

Full model:

```text
real = concat([planet_mask, fleet_mask, global_real])
key_padding_mask = ~real
```

Planet-only:

```text
real = concat([planet_mask, global_real])
key_padding_mask = ~real
```

Attention scorelarında padded key slotları `-inf` yapılır. Query padded olsa bile output numerical safety için `nan_to_num` ile korunur.


## 9. Source / No-op Head

Source head her planet token için launch logit üretir. Invalid source planetler `source_mask` ile kapatılır.

```text
src_raw = src_head(planet_tok).squeeze(-1)
src_raw = src_raw.masked_fill(~source_mask, -inf)
noop = noop_logit.expand(B, 1)
src_logits = concat([src_raw, noop], dim=1)
```

Global `act(...)` pathinde source categorical dağılımı `P_max + 1` class içerir. Son class no-op'tur.

Rollout'ta daha sık kullanılan `act_source(...)` pathinde source zaten bellidir. O source için binary distribution kurulur:

```text
launch_logits = [noop_logit, src_logit_for_this_source]
```

Bu tasarım aynı turn içinde her owned planet için ayrı launch/no-op kararı almayı sağlar.


## 10. Target Pointer Head

Target logits source-conditioned pointer attention şeklinde hesaplanır.

```text
q = tgt_query(planet_tok)
k = tgt_key(planet_tok)
tgt_logits = q @ kᵀ / sqrt(d_model)
```

Shape:

```text
tgt_logits = (B, P_max, P_max)
```

Maskler:

- Source planet kendi kendine target olamaz.
- Padded planet slotları target olamaz.

`act_source(...)` source slotunu bildiği için `tgt_logits[:, src_slot, :]` dağılımından sample/argmax alır.


## 11. Ship-Bin Head

Ship head her source planet için discrete ship fraction distribution üretir.

```text
ship_logits = ship_head(planet_tok)
shape = (B, P_max, n_ship_bins)
```

Default binler observation decode tarafında:

```text
(0.1, 0.25, 0.5, 0.75, 1.0)
```

Action decode aşamasında seçilen bin source planet'teki available ships ile çarpılıp integer ship count'a dönüşür.


## 12. Action Factorization

Global action path için policy distribution şu şekilde faktörize edilir:

```text
p(a | s) = p(src_or_noop | s)
          * p(target | s, src)
          * p(ship_bin | s, src)
```

No-op durumunda target ve ship bin terms log probability'ye eklenmez.

Per-source rollout path için:

```text
p(a_src | s, source_slot) = p(launch_or_noop | s, source_slot)
                           * p(target | s, source_slot) if launch
                           * p(ship_bin | s, source_slot) if launch
```

Bu factorization PPO re-evaluation içinde aynı şekilde tekrar hesaplanır.


## 13. `act`, `act_source`, `evaluate_source_actions`

Üç önemli API vardır.

### `act(out, deterministic=False)`

Batch başına tek global action üretir. Source categorical `P_max+1` class üzerinden sample/argmax yapar.

### `act_source(out, source_slot, deterministic=False)`

Rollout'ta kullanılan path. Source slot önceden seçilidir. Sadece o source için launch-vs-noop, target ve ship-bin kararı verir.

### `evaluate_source_actions(out, src, tgt, ship_bin, is_noop)`

PPO update sırasında buffer'da saklanan eski actionların yeni policy altındaki log probability ve entropy'sini hesaplar. Scripted curriculum actionları için de kullanılır.


## 14. Value Head

Critic value estimate PPO için return/advantage hesaplarında kullanılır.

Full modelde value head global tokenı kullanır:

```text
value = MLP(global_tok)
```

Planet-only modelde value input'u daha zengindir:

```text
planet_pooled = mean(planet_tok)
value_input = concat([global_tok, planet_pooled])
value = MLP(value_input)
```

Bu, fleet tokenları çıkarıldığı için planet setinin ortalama durumunu critic'e doğrudan vermeyi amaçlar.


## 15. PPO Update ile İlişki

PPO update sırasında buffer minibatchleri tekrar forward edilir. Sonra stored actionlar current policy altında evaluate edilir.

```text
out = policy(batch_observation)
ev = policy.evaluate_source_actions(out, src, tgt, ship_bin, is_noop)
new_logp = ev['log_prob']
entropy = ev['entropy'].mean()
new_value = ev['value']
ratio = exp(new_logp - old_logp)
policy_loss = clipped_surrogate(ratio, advantage)
value_loss = clipped_or_mse(new_value, returns)
loss = policy_loss + value_coef * value_loss - entropy_coef * entropy
```

Bu yüzden action factorization rollout ve PPO update arasında birebir tutarlı olmalıdır.


## 16. Deterministic vs Stochastic Action

Training rollout stochastic action kullanır. Replay/watch sırasında deterministic mod genellikle argmax kullanır.

Bu iki davranış ayrışabilir:

- Stochastic policy erken launch yapabilir.
- Deterministic argmax hâlâ no-op seçebilir.

Bu özellikle source launch-vs-noop logits birbirine yakınken veya noop logit yüksek kaldığında görülür. Scripted expansion curriculum bu problemi azaltmayı amaçlar ama sadece rollout behavior'u override eder; policy logits'e yeterince imitation/advantage signal gelmezse deterministic argmax geç değişebilir.


## 17. Scripted Expansion ve Model

Scripted expansion, model mimarisini değiştirmez. Rollout sırasında policy actionı üretildikten sonra, ilk `N` turn içinde uygun neutral target bulunursa action scripted launch ile override edilir.

Önemli detay: scripted action env'de gerçekten uygulanır ve `evaluate_source_actions` ile o action'ın current policy altındaki log probability'si hesaplanır. Böylece PPO o action'ın gerçek reward outcome'unu öğrenir.

Akış:

```text
policy forward
act_source sample
if scripted active: action = scripted_action
decode_action_with_launch
env.step
reward tracker attribution
PPO update with scripted action log_prob
```


## 18. Parametre Sayısı ve Training Config

Son kullanılan planet-only checkpoint yaklaşık `5,017,639` parametreye sahipti. Bu tipik olarak şu ayarlara karşılık gelir:

```text
model_variant = planet_only
d_model = 256
n_heads = 4
n_layers = 6
P_max = 48
n_ship_bins = 5
```

Küçük default train config ise `d_model=128`, `n_layers=4` kullanabilir. Checkpoint resume ederken mimari checkpoint config'inden yüklenir; CLI mimari flagleri eski checkpoint ile çelişirse dikkatli olunmalıdır.


In [ ]:
def transformer_params_rough(d_model=256, n_heads=4, n_layers=6, p_dim=33, g_dim=14, edge_dim=6):
    # Rough intuition only; exact count should use sum(p.numel() for p in model.parameters()).
    proj = p_dim*d_model + g_dim*d_model + edge_dim*n_heads
    block = 4*d_model*d_model + 2*(d_model*4*d_model)  # qkv/out + FFN rough
    heads = d_model + d_model*d_model*2 + d_model*5 + d_model*2
    return {'input_and_edge_proj_rough': proj, 'per_block_rough': block, 'heads_rough': heads, 'total_rough': proj + n_layers*block + heads}

transformer_params_rough()


## 19. Failure Modes

### No-op collapse

Belirti:

```text
H -> 0
launch/p -> 0
D -> yüksek draw sayısı
```

Sebep genellikle no-op logit'in launch logits'e göre baskın hale gelmesi, reward değişikliğinin PPO update'i destabilize etmesi veya entropy'nin hızlı çökmesidir.

### Deterministic late attack

Stochastic rollout launch yaparken deterministic replay çok geç launch yapabilir. Bu, argmax'ın no-op tarafında kalmasından kaynaklanır.

### KL explosion

Reward scale veya reward shape bir anda değişirse old/new policy ratio çok büyür. `KL > 0.1` dikkat; `KL > 1` ciddi instability işaretidir.

### Value loss spike

Delayed rewardlar veya production exponent reward range'ini büyütürse critic targetları değişir ve `valL` sıçrayabilir.


## 20. Model Değiştirirken Dikkat Edilecekler

- `planet_dim`, `fleet_dim`, `global_dim`, `edge_dim`, `d_model`, `n_layers`, `n_heads` değişiklikleri checkpoint compatibility etkiler.
- Observation feature sırası değişirse model yüklenebilir ama anlamlar bozulur; bu en tehlikeli durumdur.
- Source/target/ship factorization değiştirilirse PPO buffer, rollout ve evaluation birlikte güncellenmelidir.
- `act_source` ve `evaluate_source_actions` matematiksel olarak aynı log-prob factorization'ı kullanmalıdır.
- No-op davranışını değiştirmek için sadece reward değil, logit bias, entropy, curriculum veya auxiliary imitation loss gerekebilir.
- Deterministic watch her training değişikliğinden sonra mutlaka kontrol edilmelidir.
